# Classification with a Linear Layer

The simplest possible neural network: a single linear layer that learns to
classify 2D points into 3 classes. This is the idris-ml equivalent of a
PyTorch hello-world classifier.

**What you'll see:**
- Type-safe model construction with compile-time dimension checking
- The `runTraining` loop with a native optimizer
- Evaluation on the training data

**CLI equivalent:** `make example-supervised` (1000 epochs, full eval output)


## Architecture

A `Linear` layer maps input dimension `i` to output dimension `o` via
`y = Wx + b`. The type system enforces that the network's input and output
dimensions match the data.


In [ ]:
:t linearLayer


In [ ]:
:t Network


The `Network i hs o ty` type says: input dimension `i`, hidden layer dimensions
`hs`, output dimension `o`, element type `ty`. For a single linear layer with
no hidden layers, `hs = []`.

Let's build one that maps 2D input to 3 classes:


In [ ]:
:exec do { ll <- linearLayerAny {i=2} {o=3} "ll0";
  model <- pure ((OutputLayer ll));
  putStrLn "Model built." }


`autoName` assigns parameter names (`ll0_weight0`, `ll0_bias0`) which register
them with the C backend for gradient tracking. Without naming, parameters are
invisible to the optimizer.


## Data

Five 2D points with one-hot encoded class labels. The decision function is
`argmax(x - y - 10, -4x + y + 5, 2x + y - 11)` which creates three regions
in the plane.


In [ ]:
:t DataPoint


In [ ]:
:t TensorDataPoint


`DataPoint i o ty` is a pure Idris record with input `Vector i ty` and target
`Vector o ty`. For the C-level training path, we convert to `TensorDataPoint`
using `toTDP`, which packs the data into persistent C tensors.


## Training

We use SGD at learning rate 0.1, cross-entropy loss, and train for 500 epochs.
After training, (V2 has no `toDoubleNetwork`; evaluation uses `forwardVar` directly) converts the C-backed model to a pure Idris
model for evaluation.


_Note: this evaluation cell used V1's `toDoubleNetwork` which was removed in the Path C migration. V2 evaluates by running `forwardVar` on the trained model directly and reading scalars via `prim__item1d outT.tensorPtr <i>`. See `packages/idris-ml-examples/src/Example/Supervised.idr` for an idiomatic V2 evaluation pattern._

## Type Safety in Action

The key guarantee: if the code compiles, the dimensions match. Try changing
`{i=2, o=3}` to `{i=2, o=4}` and the data points will fail to type-check
because the one-hot targets are `Vector 3 Double`, not `Vector 4 Double`.

This is checked at compile time with zero runtime cost.


## PyTorch Comparison

The equivalent PyTorch code:

```python
model = nn.Linear(2, 3)
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(500):
    output = model(input_tensor)
    loss = F.cross_entropy(output, target_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

The main differences:
- PyTorch dimensions are checked at runtime. idris-ml checks at compile time
- PyTorch requires manual `zero_grad()` / `backward()` / `step()` sequencing.
  idris-ml fuses these in the epoch function
- PyTorch's `model.parameters()` is dynamic. idris-ml's `autoName` registers
  parameters statically

See `pytorch/torch_ref/scripts/supervised.py` for the full reference implementation.


Next: [RNN and LSTM](rnn_lstm.ipynb) — recurrent models for sequential patterns.
